In [2]:
import random, re, string
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

# ---------- paths ----------

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

IN_CSV = DATA_DIR / "Master_data_Zuzana.csv"
OUT_TXT = DATA_DIR / "Twitter_Nifty50_Corpus.txt"

# ---------- reproducibility ----------
random.seed(7)
np.random.seed(7)

# ---------- load market data ----------
df = (
    pd.read_csv(IN_CSV, parse_dates=["Date"])
      
)
df = df[(df["Date"] >= "2019-01-02") & (df["Date"] <= "2024-12-30")].copy()
df["ret"] = df["NIFTY50_Daily_Return"].fillna(0.0)

# ---------- turn daily return into tone bucket ----------
def tone_bucket(r):
    # thresholds in percent (close/close used as proxy for "open trend")
    if r >= 0.60:   return "strong_up"
    if r >= 0.15:   return "up"
    if r <= -0.60:  return "strong_down"
    if r <= -0.15:  return "down"
    return "flat"

df["tone"] = df["ret"].apply(tone_bucket)

# ---------- vocabulary / style ----------
ALIAS   = ["Nifty 50", "Nifty50", "#NIFTY50"]      # no quote chars → avoids stray ' in wordclouds
TICKERS = ["$NIFTY", "$NSE"]
TAGS    = ["#India", "#Sensex", "#Markets", "#DalalStreet"]

EMOJIS_POS = ["🚀", "📈", "🔥", "🤑", "✅", "💚"]
EMOJIS_NEG = ["📉", "😬", "❌", "🥶", "🔻", "💣"]
EMOJIS_FLAT= ["🤔", "🧭", "⏳", "🤝", "📰"]

MENTIONS   = ["@traderjoe", "@quantgal", "@macroalpha", "@riskparity", "@valuehacker", "@newsbot"]
URL_BASES  = ["https://t.co/", "http://bit.ly/"]

# stock-tweet verbs & add-ons
OPEN_UP     = ["gaps up", "opens green", "pops at the open", "pushes higher off the bell", "ticks up on the open"]
OPEN_DOWN   = ["gaps down", "opens red", "slips at the open", "pushes lower off the bell", "ticks down on the open"]
OPEN_FLAT   = ["opens flat", "quiet open", "unch at the bell", "sideways open", "basically flat open"]

ADD_BULL    = ["buyers active", "breadth positive", "dip bought", "breakout watch", "risk on", "follow-through building", "new highs in sight"]
ADD_BEAR    = ["sellers in control", "breadth weak", "sell the rip?", "breakdown watch", "risk off", "lower highs", "failed bounce"]
ADD_NEUTRAL = ["range day?", "inside day vibes", "waiting on data", "no big flows yet", "two-way trade", "patience"]

# explicit sentiment words (guaranteed insertion)
POS_WORDS = ["bullish", "constructive", "positive", "accumulating", "bid", "support respected", "breakout"]
NEG_WORDS = ["bearish", "negative", "distribution", "offer heavy", "rejected", "breakdown", "supply"]

MACRO = ["CPI print", "PMI", "GDP", "earnings", "rates", "yields", "crude", "FX", "geopolitics", "bond supply"]

def rand_url():
    base = random.choice(URL_BASES)
    slug = "".join(random.choices(string.ascii_letters + string.digits, k=random.randint(6, 10)))
    return base + slug

def times_near_open(n):
    # India cash open ~09:15 IST. Heavily bias 08:45–11:30, otherwise 07:30–15:30.
    out = []
    for _ in range(n):
        if random.random() < 0.72:
            h = random.randint(8, 11)
        else:
            h = random.randint(7, 15)
        m = random.randint(0, 59)
        out.append(f"{h:02d}{m:02d}")
    return out

def choose_alias():  return random.choice(ALIAS)
def choose_ticker(): return random.choice(TICKERS)
def choose_tags(k=2): return " ".join(random.sample(TAGS, k=random.randint(1, k)))

def build_line(date_str, hhmm, tone):
    alias  = choose_alias()
    ticker = choose_ticker()
    tags   = choose_tags(2)
    macro  = random.choice(MACRO)
    handle = random.choice(MENTIONS) if random.random() < 0.2 else ""
    url    = rand_url() if random.random() < 0.45 else ""

    if tone == "strong_up":
        verb = random.choice(OPEN_UP)
        adds = random.sample(ADD_BULL, k=2)
        senti = random.sample(POS_WORDS, k=2)
        emj  = " " + random.choice(EMOJIS_POS)
        lead = random.choice(["Morning take:", "Heads up:"])
    elif tone == "up":
        verb = random.choice(OPEN_UP)
        adds = random.sample(ADD_BULL, k=1) + (random.sample(ADD_NEUTRAL, k=1) if random.random() < 0.5 else [])
        senti = random.sample(POS_WORDS, k=1)
        emj  = " " + random.choice(EMOJIS_POS + EMOJIS_FLAT)
        lead = random.choice(["Quick take:", "Note:"])
    elif tone == "strong_down":
        verb = random.choice(OPEN_DOWN)
        adds = random.sample(ADD_BEAR, k=2)
        senti = random.sample(NEG_WORDS, k=2)
        emj  = " " + random.choice(EMOJIS_NEG)
        lead = random.choice(["Caution:", "Heads up:"])
    elif tone == "down":
        verb = random.choice(OPEN_DOWN)
        adds = random.sample(ADD_BEAR, k=1) + (random.sample(ADD_NEUTRAL, k=1) if random.random() < 0.5 else [])
        senti = random.sample(NEG_WORDS, k=1)
        emj  = " " + random.choice(EMOJIS_NEG + EMOJIS_FLAT)
        lead = random.choice(["Quick take:", "Note:"])
    else:  # flat
        verb = random.choice(OPEN_FLAT)
        adds = random.sample(ADD_NEUTRAL, k=2 if random.random() < 0.5 else 1)
        senti = random.sample(POS_WORDS + NEG_WORDS, k=1)
        emj  = " " + random.choice(EMOJIS_FLAT)
        lead = random.choice(["Morning:", "EOD:"])  # sometimes mislabeled on purpose like real TL

    # compose tweet (keep it ≤ ~280 chars naturally)
    bits = [
        f"[{date_str} {hhmm}]",
        f"{lead}",
        f"{alias} {verb}.",
        f"{macro}.",
        " ".join(adds + senti),
        tags,
        ticker + emj,
        handle,
        url,
    ]
    return " ".join(x for x in bits if x).strip()

# ---------- generate ----------
MEAN_PER_DAY = 60   # tweak up/down to change corpus size
tweets = []
for _, row in df.iterrows():
    n = np.random.poisson(lam=MEAN_PER_DAY)
    n = max(n, 1)
    dstr = row["Date"].strftime("%Y-%m-%d")
    for t in times_near_open(n):
        tweets.append(build_line(dstr, t, row["tone"]))

# ---------- write ----------
out_path = Path(OUT_TXT)
with out_path.open("w", encoding="utf-8", newline="\n") as f:
    for line in tweets:
        f.write(line + "\n")

print(f"wrote {len(tweets):,} lines")


wrote 93,819 lines
